In [ ]:
import getpass
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal
import os
import time
import getpass



api_key = getpass.getpass("Enter your OpenAI API Key: ")
client = OpenAI(api_key=api_key)



Enter your OpenAI API Key: ··········


In [ ]:
class ReviewExtraction(BaseModel):
    root_cause: Literal["Sizing / Fit",
        "Fabric / Material Quality",
        "Design / Cut Mismatch",
        "Defective Workmanship",
        "Customer Expectation",
        "Positive / No Issue"]
    specific_component: str
    fit_issue: Literal["Runs Small", "Runs Large", "Too Long", "Too Short", "Not Applicable"]
    return_severity: Literal[1,2,3]  #1 = Minor severity to 3= unwearable


In [ ]:
#connection test and api key test
test_review = "Title: Broken Zipper\nReview: The jacket looked great, but the zipper snapped on day two. Super disappointed."
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a apparel quality assurance analyst. Extract structured return defect tags from clothing reviews."},
        {"role": "user", "content": test_review},],response_format=ReviewExtraction,)
parsed_data = completion.choices[0].message.parsed


In [ ]:


raw_df = pd.read_csv("Womens_Clothing_E-Commerce_Reviews.csv", index_col = 0)

#keep reviews only Rating <= 2
reviews_to_process = (raw_df[(raw_df["Rating"] <= 2) & (raw_df["Review Text"].notna())].reset_index(drop=True))
total_rows = len(reviews_to_process)
print(f"Total negative reviews: {total_rows}")


#sends row into gpt-mini to parse, return a dict of the root cause, specific component with a problem, fit issue if applicable and severity of return
def parse_review(row):
    title = str(row["Title"]) if pd.notna(row["Title"]) else "No Title"
    body = str(row["Review Text"])
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a retail QA analyst. Extract structured defect and return data from customer clothing reviews."
                },
                {"role": "user", "content": f"Title: {title}\nReview: {body}"}
            ],
            response_format=ReviewExtraction,
        )
        return completion.choices[0].message.parsed.model_dump()
    except Exception as e:
        return {
            "root_cause": "Other",
            "specific_component": "Error",
            "fit_issue": "Not Applicable",
            "return_severity": 0
        }


#checkpoint
checkpoint = "checkpoint_records.csv"
if os.path.exists(checkpoint):
    checkpoint_df = pd.read_csv(checkpoint)
    parsed_records = checkpoint_df.to_dict(orient="records")
    start_index = len(parsed_records)
else:
    parsed_records = []
    start_index = 0


#batch extraction loop, save checkpoint every 50 rows
start_time = time.time()
for i in range(start_index, total_rows):
    row = reviews_to_process.iloc[i]
    parsed_records.append(parse_review(row))
    if (i + 1) % 50 == 0 or (i + 1) == total_rows:
        elapsed = round((time.time() - start_time) / 60, 1)
        print(f"Processed {i + 1} / {total_rows} reviews ({elapsed} mins elapsed)")
        pd.DataFrame(parsed_records).to_csv(checkpoint, index=False)


parsed_df = pd.DataFrame(parsed_records) #join all dicts into df
merged_df = pd.concat([reviews_to_process, parsed_df], axis=1) #merge original review with gpt-mini defect columns


if os.path.exists(checkpoint):
    os.remove(checkpoint)
print(f"finished, {total_rows} rows")

Total negative reviews to process: 2370
Starting batch processing...
Processed 50 / 2370 reviews (0.8 mins elapsed)...
Processed 100 / 2370 reviews (1.6 mins elapsed)...
Processed 150 / 2370 reviews (2.3 mins elapsed)...
Processed 200 / 2370 reviews (3.0 mins elapsed)...
Processed 250 / 2370 reviews (3.8 mins elapsed)...
Processed 300 / 2370 reviews (4.6 mins elapsed)...
Processed 350 / 2370 reviews (5.3 mins elapsed)...
Processed 400 / 2370 reviews (6.1 mins elapsed)...
Processed 450 / 2370 reviews (6.8 mins elapsed)...
Processed 500 / 2370 reviews (7.6 mins elapsed)...
Processed 550 / 2370 reviews (8.3 mins elapsed)...
Processed 600 / 2370 reviews (9.1 mins elapsed)...
Processed 650 / 2370 reviews (9.8 mins elapsed)...
Processed 700 / 2370 reviews (10.6 mins elapsed)...
Processed 750 / 2370 reviews (11.3 mins elapsed)...
Processed 800 / 2370 reviews (12.1 mins elapsed)...
Processed 850 / 2370 reviews (12.7 mins elapsed)...
Processed 900 / 2370 reviews (13.5 mins elapsed)...
Processed

NameError: name 'create_engine' is not defined

In [ ]:
enriched_df.to_csv("Filtered_women_clothing_dataset.csv", index=False)